# 01 — Interactive Model Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chris-L6/causalfm-survey/blob/main/notebooks/01_interactive_model_demo.ipynb)

**Choose a model and dataset, then run it!**

This notebook lets you interactively select from:
- **Models**: Foundation models (CausalPFN, Do-PFN, CausalFM) or traditional metalearners (S/T/X-learner, Debiased ML, IPW, DR)
- **Datasets**: Synthetic (linear, nonlinear, IV, frontdoor) or real-world (Lalonde)

All models follow a unified interface, so swapping between them is seamless.

## 1. Environment Setup

If running on **Colab**, this clones the repo. If **local**, it imports from parent directory.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

# ── FOR COLAB ONLY: set your GitHub token if the repo is private ──────────────
# Create one at: github.com/settings/tokens  (scope: repo → read)
# Leave as "" if the repo is public.
GITHUB_TOKEN = ""
# ──────────────────────────────────────────────────────────────────────────────

REPO_SLUG = "chris-L6/causalfm-survey"
REPO_DIR  = "causalfm-survey"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        if GITHUB_TOKEN:
            clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"
        else:
            clone_url = f"https://github.com/{REPO_SLUG}.git"
        result = subprocess.run(["git", "clone", clone_url], capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(
                "git clone failed — the repo is likely private.\n"
                "Fix: set GITHUB_TOKEN above (github.com/settings/tokens, scope: repo→read).\n"
                f"Error: {result.stderr.strip()}"
            )
    sys.path.insert(0, REPO_DIR)
else:
    sys.path.insert(0, os.path.abspath(".."))

import causal_bench
print("causal_bench imported from:", causal_bench.__file__)

In [ ]:
%pip install -q econml causalpfn
import os, numpy as np, pandas as pd, time, warnings
warnings.filterwarnings('ignore')

import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device: {device}")

## 2. Dataset Loader

In [15]:
from causal_bench import get_dataset, load_lalonde, list_available_datasets

# Load all datasets
DATASETS = {}
print("Loading synthetic datasets...")
for name in ["linear_confounded", "nonlinear_heterogeneous", "iv_binary", "frontdoor"]:
    DATASETS[name] = get_dataset(name, n=2000, seed=0)

print("Loading Lalonde dataset...")
try:
    DATASETS["lalonde_nsw_psid"] = load_lalonde()
    print("  ✓ Lalonde loaded")
except Exception as e:
    print(f"  ✗ Lalonde unavailable: {e}")

print(f"\nAvailable datasets: {list(DATASETS.keys())}")
for name, ds in DATASETS.items():
    print(f"  {name}: n={len(ds.Y)}, X.shape={ds.X.shape}, ATE={ds.ate:.3f}")

Loading synthetic datasets...
Loading Lalonde dataset...
  ✓ Lalonde loaded

Available datasets: ['linear_confounded', 'nonlinear_heterogeneous', 'iv_binary', 'frontdoor', 'lalonde_nsw_psid']
  linear_confounded: n=2000, X.shape=(2000, 5), ATE=2.000
  nonlinear_heterogeneous: n=2000, X.shape=(2000, 6), ATE=0.971
  iv_binary: n=2000, X.shape=(2000, 5), ATE=1.500
  frontdoor: n=2000, X.shape=(2000, 5), ATE=2.200
  lalonde_nsw_psid: n=2675, X.shape=(2675, 8), ATE=-15204.777


## 3. Select Model & Dataset

**Change `SELECTED_MODEL` and `SELECTED_DATASET` below, then run this cell + the next one.**

In [17]:
# ── EDIT THESE TWO LINES ──────────────────────────────────────────────────────
SELECTED_MODEL   = "X-learner"               # Foundation : CausalPFN* | Do-PFN | CausalFM
                                              # Metalearner: S-learner | T-learner | X-learner
                                              #              Debiased ML | IPW | DR (Doubly Robust)
SELECTED_DATASET = "nonlinear_heterogeneous"  # linear_confounded | nonlinear_heterogeneous
                                              # iv_binary | frontdoor | lalonde_nsw_psid
# * CausalPFN crashes on Apple Silicon (torch.compile/CUDA artifact). Use metalearners for now.
# ──────────────────────────────────────────────────────────────────────────────

from causal_bench import (
    CausalPFNWrapper, DoPFNWrapper, CausalFMWrapper,
    SLearnerWrapper, TLearnerWrapper, XLearnerWrapper,
    DebiasedMLWrapper, IPWWrapper, DRWrapper,
)

MODELS = {
    "CausalPFN":          CausalPFNWrapper,
    "Do-PFN":             DoPFNWrapper,
    "CausalFM":           CausalFMWrapper,
    "S-learner":          SLearnerWrapper,
    "T-learner":          TLearnerWrapper,
    "X-learner":          XLearnerWrapper,
    "Debiased ML":        DebiasedMLWrapper,
    "IPW":                IPWWrapper,
    "DR (Doubly Robust)": DRWrapper,
}

print("Model availability:")
for name, cls in MODELS.items():
    print(f"  {'✓' if cls.is_available() else '✗'}  {name}")

assert SELECTED_MODEL in MODELS,    f"Unknown model {SELECTED_MODEL!r}. Options: {list(MODELS)}"
assert SELECTED_DATASET in DATASETS, f"Unknown dataset {SELECTED_DATASET!r}. Options: {list(DATASETS)}"
print(f"\nReady → model={SELECTED_MODEL!r}   dataset={SELECTED_DATASET!r}")

Model availability:
  ✓  CausalPFN
  ✗  Do-PFN
  ✗  CausalFM
  ✓  S-learner
  ✓  T-learner
  ✓  X-learner
  ✓  Debiased ML
  ✓  IPW
  ✓  DR (Doubly Robust)

Ready → model='X-learner'   dataset='nonlinear_heterogeneous'


## 4. Run

In [ ]:
from causal_bench import evaluate_cate

# Guard: CausalPFN segfaults on Apple Silicon (torch.compile CUDA artifact)
KNOWN_APPLE_SILICON_CRASH = {"CausalPFN"}
if SELECTED_MODEL in KNOWN_APPLE_SILICON_CRASH and device in ("cpu", "mps"):
    print(f"⚠️  {SELECTED_MODEL} crashes on Apple Silicon (M-series Mac).")
    print("   Cause: checkpoint compiled with torch.compile for CUDA — cannot run on MPS/CPU.")
    print("   Fix: use a metalearner instead, or run on a CUDA machine.")
    raise SystemExit(0)

print(f"\n{'='*60}")
print(f"  Model  : {SELECTED_MODEL}")
print(f"  Dataset: {SELECTED_DATASET}")
print(f"{'='*60}\n")

ds = DATASETS[SELECTED_DATASET]
train_idx, test_idx = ds.train_test_split(0.7, seed=0)
X_train, X_test = ds.X[train_idx], ds.X[test_idx]
T_train, Y_train = ds.T[train_idx], ds.Y[train_idx]
tau_test = ds.tau[test_idx] if hasattr(ds, "tau") else None

model_cls = MODELS[SELECTED_MODEL]
if not model_cls.is_available():
    raise RuntimeError(f"{SELECTED_MODEL} is not installed. See README for setup instructions.")

try:
    t0 = time.time()

    if SELECTED_MODEL == "CausalFM":
        checkpoint_path = "CausalFM-toolkit/checkpoints/best_model.pth"
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"CausalFM checkpoint not found at: {checkpoint_path}")
        model = model_cls(checkpoint_path=checkpoint_path, device=device)
    else:
        model = model_cls(device=device)

    model.fit(X_train, T_train, Y_train)
    tau_hat, lower, upper = model.predict(X_test)
    runtime = time.time() - t0
    ate_hat = float(np.mean(tau_hat))

    if tau_test is not None:
        result = evaluate_cate(tau_hat, tau_test, ate_hat=ate_hat, ate_true=ds.ate,
                               lower=lower, upper=upper, runtime_s=runtime)
        print(f"{'Metric':<22} {'Value':>10}")
        print("-" * 34)
        for key, val in result.items():
            if val is not None:
                print(f"  {key:<20} {val:>10.4f}")
    else:
        print(f"  ATE_hat              {ate_hat:>10.1f}")
        print(f"  ATE_true (observed)  {ds.ate:>10.1f}")
        print(f"  ATE_abs_error        {abs(ate_hat - ds.ate):>10.1f}")
        print(f"  runtime_s            {runtime:>10.2f}")
        print("\n  (No ground-truth CATE for real-world data)")

except Exception as e:
    import traceback
    print(f"ERROR running {SELECTED_MODEL}:")
    traceback.print_exc()

## Notes

- **Foundation models** (CausalPFN, Do-PFN, CausalFM) use in-context learning: they don't re-train on your data, just condition on it.
- **Metalearners** (S/T/X, Debiased ML, etc.) are traditional ML methods trained separately for control and treatment.
- **Synthetic datasets** have ground-truth CATE (`tau`), so full metrics (PEHE, bias, coverage) are computed.
- **Lalonde** is real-world data: no ground truth CATE, only observed ATE is available.
- **Missing models** (shown with ✗) require additional installs; check the notebook setup cells.